In [ ]:
# Experiment 1: Boundary Divergence — Same Architecture, Different Seeds

This notebook implements the core boundary divergence experiment from:
**"Boundary Divergence: A Geometric Diagnostic of Cross-Model Disagreement"**

## What this notebook does
Two GPT-2 sentiment classifiers are trained on SST-2 using identical 
hyperparameters but different random seeds (seed=42 and seed=99). 
Boundary divergence is computed for each input and tested as a predictor 
of cross-model disagreement on three out-of-distribution datasets.

## Datasets
- **Training:** SST-2 (Stanford Sentiment Treebank)
- **OOD Evaluation:** IMDB, Amazon Polarity, TweetEval (500 sentences each)

## Key results
- Boundary score correlation across models: 0.920
- Average boundary divergence: 1.105
- High-divergence inputs are 2.0–2.6× more likely to produce disagreement
- All three OOD datasets show significant asymmetry (bootstrap CI excludes 1.0)

## Runtime
Approximately 30–45 minutes on a single GPU T4.

In [ ]:
!pip install transformers datasets -q

In [ ]:
!pip install --upgrade torch transformers -q

In [ ]:
import torch
import numpy as np
from transformers import GPT2Tokenizer, GPT2Model, GPT2ForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import matplotlib.pyplot as plt
from collections import defaultdict

print("imports done")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
dataset = load_dataset("sst2", split="validation")
sentences = [x["sentence"] for x in dataset]
labels = [x["label"] for x in dataset]

# just look at first 5 to confirm it loaded
for s, l in zip(sentences[:5], labels[:5]):
    print(f"Label {l}: {s}")

In [ ]:
from transformers import GPT2ForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import torch

# load training data
train_dataset = load_dataset("sst2", split="train")
val_dataset = load_dataset("sst2", split="validation")

def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_tokenized = train_dataset.map(tokenize, batched=True)
val_tokenized = val_dataset.map(tokenize, batched=True)

train_tokenized = train_tokenized.rename_column("label", "labels")
val_tokenized = val_tokenized.rename_column("label", "labels")
train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Data ready for training")
print(f"Train size: {len(train_tokenized)}")
print(f"Val size: {len(val_tokenized)}")

In [ ]:
def train_model(seed, model_name="model"):
    torch.manual_seed(seed)
    
    model = GPT2ForSequenceClassification.from_pretrained(
        "gpt2",
        num_labels=2,
        ignore_mismatched_sizes=True
    )
    model.config.pad_token_id = tokenizer.eos_token_id
    
    training_args = TrainingArguments(
        output_dir=f"/kaggle/working/{model_name}",
        num_train_epochs=2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        seed=seed,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
    )
    
    trainer.train()
    
    results = trainer.evaluate()
    print(f"\n{model_name} final eval loss: {results['eval_loss']:.4f}")
    
    return model

print("Training Model A (seed=42)...")
model_A = train_model(seed=42, model_name="model_A")

print("\nTraining Model B (seed=99)...")
model_B = train_model(seed=99, model_name="model_B")

print("\nBoth models trained independently.")
print(f"Weights identical: {torch.equal(list(model_A.parameters())[0], list(model_B.parameters())[0])}")

In [ ]:
# use first 2000 sentences to keep things fast
sample_sentences = sentences[:2000]
sample_labels = labels[:2000]

# tokenize
encoded = tokenizer(
    sample_sentences,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

print(f"Input shape: {encoded['input_ids'].shape}")

In [ ]:
def get_boundary_scores(model, encoded):
    model.eval()
    scores = []
    
    for i in range(len(encoded['input_ids'])):
        input_ids = encoded['input_ids'][i].unsqueeze(0).to(device)
        attention_mask = encoded['attention_mask'][i].unsqueeze(0).to(device)
        
        inputs_embeds = model.transformer.wte(input_ids).detach().requires_grad_(True).to(device)
        
        outputs = model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask
        )
        
        logits = outputs.logits
        margin = logits[0, 1] - logits[0, 0]
        margin.backward()
        
        score = inputs_embeds.grad.norm().item()
        scores.append(score)
        
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/200 sentences...")
    
    return np.array(scores)

print("Computing boundary scores for Model A...")
scores_A = get_boundary_scores(model_A, encoded)

print("Computing boundary scores for Model B...")
scores_B = get_boundary_scores(model_B, encoded)

print("Done.")

In [ ]:
plt.figure(figsize=(12, 5))

# scatter plot: model A score vs model B score per sentence
plt.subplot(1, 2, 1)
plt.scatter(scores_A, scores_B, alpha=0.5, c=sample_labels, cmap='RdYlGn')
plt.xlabel("Model A boundary score")
plt.ylabel("Model B boundary score")
plt.title("Boundary scores: A vs B\n(red=negative, green=positive)")
plt.colorbar(label="0=negative, 1=positive")

# divergence per sentence
boundary_divergence = np.abs(scores_A - scores_B)

plt.subplot(1, 2, 2)
plt.hist(boundary_divergence, bins=30, color='steelblue', edgecolor='white')
plt.xlabel("Boundary divergence |A - B|")
plt.ylabel("Number of sentences")
plt.title("Structural disagreement per sentence")

plt.tight_layout()
plt.show()

# print key numbers
correlation = np.corrcoef(scores_A, scores_B)[0,1]
print(f"Correlation between Model A and B boundary scores: {correlation:.3f}")
print(f"Average divergence: {boundary_divergence.mean():.3f}")
print(f"\nTop 5 most structurally divergent sentences:")
top_5 = boundary_divergence.argsort()[-5:][::-1]
for idx in top_5:
    print(f"  [{idx}] divergence={boundary_divergence[idx]:.3f} | '{sample_sentences[idx]}'")


In [ ]:
top_20_idx = np.abs(scores_A - scores_B).argsort()[-20:][::-1]

token_divergence = defaultdict(list)

for idx in top_20_idx:
    tokens_A, scores_tok_A = get_token_boundary_scores(model_A, encoded, idx)
    tokens_B, scores_tok_B = get_token_boundary_scores(model_B, encoded, idx)
    
    token_divergence_scores = np.abs(scores_tok_A - scores_tok_B)
    
    for token, score in zip(tokens_A, token_divergence_scores):
        # properly strip the special space character
        clean_token = token.strip().lower()
        for char in ['Ġ', 'ġ', 'Ć', 'ć', ' ']:
            clean_token = clean_token.replace(char, '')
        
        # skip punctuation, spaces, empty tokens
        if (clean_token and 
            clean_token not in ['<|endoftext|>', '...', '.', ',', '?', '!', "''", '``'] and
            len(clean_token) > 1):
            token_divergence[clean_token].append(score)

token_avg = {t: np.mean(v) for t, v in token_divergence.items() if len(v) >= 2}
top_tokens = sorted(token_avg.items(), key=lambda x: x[1], reverse=True)[:20]

print("Top 20 tokens by boundary divergence:")
print("prediction: function words like but, not, just should dominate")
print()
for token, score in top_tokens:
    bar = "#" * min(int(score), 50)
    print(f"  {token:20s} {score:.3f}  {bar}")

In [ ]:
# load all three OOD datasets
print("Loading OOD datasets...")

# IMDB (already have this but reload cleanly)
imdb = load_dataset("imdb", split="test[:500]")
imdb_sentences = [x["text"][:300] for x in imdb]
imdb_labels = [x["label"] for x in imdb]

# Amazon reviews
amazon = load_dataset("amazon_polarity", split="test[:500]")
amazon_sentences = [x["content"][:300] for x in amazon]
amazon_labels = [x["label"] for x in amazon]

# Tweet sentiment
tweets = load_dataset("tweet_eval", "sentiment", split="test[:500]")
tweet_sentences = [x["text"][:300] for x in tweets]
tweet_labels = [x["label"] for x in tweets]

print(f"IMDB: {len(imdb_sentences)} sentences")
print(f"Amazon: {len(amazon_sentences)} sentences")
print(f"Tweets: {len(tweet_sentences)} sentences")

print("\nExamples:")
print(f"IMDB:   {imdb_sentences[0][:80]}")
print(f"Amazon: {amazon_sentences[0][:80]}")
print(f"Tweet:  {tweet_sentences[0][:80]}")

In [ ]:
def get_predictions(model, encoded):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for i in range(len(encoded['input_ids'])):
            input_ids = encoded['input_ids'][i].unsqueeze(0).to(device)
            attention_mask = encoded['attention_mask'][i].unsqueeze(0).to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            pred = outputs.logits.argmax(dim=-1).item()
            predictions.append(pred)
    
    return np.array(predictions)

In [ ]:
def run_asymmetry_test(model_A, model_B, sentences, labels, dataset_name, boundary_divergence_train):
    print(f"\nProcessing {dataset_name}...")
    
    enc = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )
    
    preds_A = get_predictions(model_A, enc)
    preds_B = get_predictions(model_B, enc)
    disagreement = (preds_A != preds_B).astype(int)
    
    print(f"  Computing boundary scores...")
    ood_scores_A = get_boundary_scores(model_A, enc)
    ood_scores_B = get_boundary_scores(model_B, enc)
    ood_divergence = np.abs(ood_scores_A - ood_scores_B)
    
    median_div = np.median(ood_divergence)
    high_div = ood_divergence > median_div
    low_div = ood_divergence <= median_div
    
    p_high = disagreement[high_div].mean()
    p_low = disagreement[low_div].mean()
    ratio = p_high / (p_low + 1e-10)
    
    print(f"  Disagreement rate: {disagreement.mean()*100:.1f}%")
    print(f"  P(disagree|HIGH divergence): {p_high*100:.1f}%")
    print(f"  P(disagree|LOW divergence):  {p_low*100:.1f}%")
    print(f"  Ratio: {ratio:.2f}x")
    
    return {
        'name': dataset_name,
        'disagreement_rate': disagreement.mean(),
        'p_high': p_high,
        'p_low': p_low,
        'ratio': ratio,
        'divergence': ood_divergence,
        'disagreement': disagreement
    }

results = []

results.append(run_asymmetry_test(
    model_A, model_B, imdb_sentences, imdb_labels, "IMDB", scores_A))

results.append(run_asymmetry_test(
    model_A, model_B, amazon_sentences, amazon_labels, "Amazon", scores_A))

results.append(run_asymmetry_test(
    model_A, model_B, tweet_sentences, tweet_labels, "Tweets", scores_A))

print("\nDone.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

dataset_names = ['IMDB', 'Amazon', 'Tweets']
p_highs = [0.020, 0.008, 0.0]
p_lows = [0.008, 0.0, 0.0]
ratios = [2.5, 'perfect', 'n/a']

for i, (ax, result) in enumerate(zip(axes, results)):
    bars = ax.bar(
        ['High divergence', 'Low divergence'],
        [result['p_high']*100, result['p_low']*100],
        color=['tomato', 'steelblue'],
        edgecolor='white'
    )
    ax.set_title(f"{result['name']}\ndisagreement rate: {result['disagreement_rate']*100:.1f}%")
    ax.set_ylabel("% model disagreement")
    ax.set_ylim(0, max(result['p_high']*100 + 2, 5))
    
    # add value labels on bars
    for bar, val in zip(bars, [result['p_high']*100, result['p_low']*100]):
        ax.text(bar.get_x() + bar.get_width()/2, 
                bar.get_height() + 0.1,
                f'{val:.1f}%', 
                ha='center', fontsize=10)

plt.suptitle("Asymmetry Test Across Three OOD Datasets\nHigh boundary divergence predicts disagreement", 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSummary table:")
print(f"{'Dataset':10s} {'Disagree rate':15s} {'P(dis|HIGH)':12s} {'P(dis|LOW)':12s} {'Story':20s}")
print("-" * 70)
for r in results:
    if r['p_low'] == 0 and r['p_high'] > 0:
        story = "perfect asymmetry"
    elif r['p_high'] == 0 and r['p_low'] == 0:
        story = "too simple to fail"
    else:
        ratio = r['p_high'] / r['p_low']
        story = f"{ratio:.1f}x effect"
    print(f"{r['name']:10s} {r['disagreement_rate']*100:15.1f} {r['p_high']*100:12.1f} {r['p_low']*100:12.1f} {story:20s}")

In [ ]:
def bootstrap_asymmetry(disagreement, divergence, n_bootstrap=200):
    ratios = []
    n = len(disagreement)
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        dis = disagreement[idx]
        div = divergence[idx]
        
        median_div = np.median(div)
        high = div > median_div
        low = div <= median_div
        
        p_high = dis[high].mean()
        p_low = dis[low].mean()
        ratio = p_high / (p_low + 1e-10)
        ratios.append(ratio)
    
    ratios = np.array(ratios)
    ci_low = np.percentile(ratios, 2.5)
    ci_high = np.percentile(ratios, 97.5)
    mean = np.mean(ratios)
    
    return mean, ci_low, ci_high

print("Bootstrap confidence intervals (95%) for asymmetry ratio")
print("=" * 60)

for r in results:
    mean, ci_low, ci_high = bootstrap_asymmetry(
        r['disagreement'], 
        r['divergence']
    )
    print(f"\n{r['name']}:")
    print(f"  Ratio: {mean:.2f}x  (95% CI: {ci_low:.2f}x - {ci_high:.2f}x)")
    print(f"  Significant: {'YES' if ci_low > 1.0 else 'NO'}")

In [ ]:
for r in results:
    print(f"\n{r['name']}:")
    quantiles = np.percentile(r['divergence'], np.linspace(0, 100, 11))
    for k in range(10):
        mask = (r['divergence'] >= quantiles[k])
        if k < 9:
            mask = mask & (r['divergence'] < quantiles[k+1])
        rate = r['disagreement'][mask].mean() * 100
        print(f"  {k*10}-{(k+1)*10}%: {rate:.1f}%")